PHẦN 1:TRẮC NGHIỆM

*CÂU 1:  Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần
mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)*

Bước 1: Tải dữ liệu lên

In [1]:
from google.colab import files
uploaded = files.upload()

Saving orders.csv to orders.csv


In [22]:
import pandas as pd

df_raw = pd.read_csv('orders.csv')  # bản gốc, không đụng vào

# Làm việc với bản copy
df = df_raw.copy()

In [23]:
import pandas as pd

df = pd.read_csv('orders.csv')

import pandas as pd

df = pd.read_csv('orders.csv')
print(f"Ban đầu: {len(df)} dòng")

# Bước 1: Bỏ null
df = df.dropna(subset=['order_date', 'customer_id'])
print(f"Sau bước 1 (dropna): {len(df)} dòng")

# Bước 2: Bỏ đơn cancelled và created (nếu có), giữ returned
df = df[~df['order_status'].isin(['cancelled', 'created'])]
print(f"Sau bước 2 (bỏ cancelled và created): {len(df)} dòng")



# Bước 3: Drop duplicate cặp (order_id, customer_id)
df = df.drop_duplicates(subset=['order_id', 'customer_id'], keep='first')
print(f"Sau bước 3 (dedup): {len(df)} dòng")

# Bước 4: Lọc customer có > 1 đơn
order_counts = df.groupby('customer_id')['order_id'].count()
valid_customers = order_counts[order_counts > 1].index
df = df[df['customer_id'].isin(valid_customers)]
print(f"Sau bước 4 (>1 đơn): {len(df)} dòng")

# Bước 5: Sort theo (customer_id, order_date)
df['order_date'] = pd.to_datetime(df['order_date'])
df = df.sort_values(['customer_id', 'order_date']).reset_index(drop=True)

# Bước 6: Tính gap từng cặp liên tiếp trong mỗi customer
df['prev_date'] = df.groupby('customer_id')['order_date'].shift(1)
df['gap_days'] = (df['order_date'] - df['prev_date']).dt.days
gaps = df.dropna(subset=['gap_days'])['gap_days']
print(f"\nTổng số gap tính được: {len(gaps)}")

# Bước 7: Sort dãy gap tăng dần
gaps_sorted = gaps.sort_values().reset_index(drop=True)

# Bước 8: Tìm median thủ công
n = len(gaps_sorted)
if n % 2 == 1:
    median_gap = gaps_sorted[n // 2]
else:
    median_gap = (gaps_sorted[n // 2 - 1] + gaps_sorted[n // 2]) / 2

print(f"Median inter-order gap: {median_gap} ngày")

# Verify bằng pandas
print(f"Verify bằng pandas median: {gaps.median()} ngày")

Ban đầu: 646945 dòng
Sau bước 1 (dropna): 646945 dòng
Sau bước 2 (bỏ cancelled và created): 580208 dòng
Sau bước 3 (dedup): 580208 dòng
Sau bước 4 (>1 đơn): 557492 dòng

Tổng số gap tính được: 492369
Median inter-order gap: 158.0 ngày
Verify bằng pandas median: 158.0 ngày


Bài 2: Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp
trung bình cao nhất, với công thức (price −cogs)/price?

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving products.csv to products (3).csv


In [ ]:
import pandas as pd

df_raw1 = pd.read_csv('orders.csv')  # bản gốc, không đụng vào

# Làm việc với bản copy
df = df_raw1.copy()

In [ ]:
import pandas as pd

df = pd.read_csv('products.csv')
print(df.columns)
print(df.head())

Index(['product_id', 'product_name', 'category', 'segment', 'size', 'color',
       'price', 'cogs'],
      dtype='object')
   product_id      product_name    category   segment size   color  \
0         536  SaigonFlex UC-01  Streetwear  Everyday    S   green   
1         537  SaigonFlex UC-02  Streetwear  Everyday    M  silver   
2         538  SaigonFlex UC-03  Streetwear  Everyday    L    pink   
3         539  SaigonFlex UC-04  Streetwear  Everyday   XL  yellow   
4         540  SaigonFlex UC-05  Streetwear  Everyday    S     red   

          price          cogs  
0  11059.650000   9704.842875  
1   9523.076013   5393.870254  
2  15951.633158  11371.919278  
3  15753.717299   8573.172954  
4  15766.334536  14063.570406  


In [ ]:

print(f"[1] Dữ liệu gốc: {len(df)} dòng")
print(df.isnull().sum())

# Kiểm tra vi phạm ràng buộc cogs < price
violation = df[df['cogs'] >= df['price']]
print(f"\n[2] Dòng vi phạm cogs >= price: {len(violation)}")

# Nếu > 0 thì mới drop
if len(violation) > 0:
    print(violation)
    df = df[df['cogs'] < df['price']]
    print(f"Sau khi drop vi phạm: {len(df)} dòng")

# Drop thiếu dữ liệu các cột cần thiết
df = df.dropna(subset=['segment', 'price', 'cogs'])
print(f"\n[3] Sau dropna: {len(df)} dòng")

# Tính gross margin
df['gross_margin'] = (df['price'] - df['cogs']) / df['price']

result = df.groupby('segment')['gross_margin'].mean().sort_values(ascending=False)
print(f"\n[4] Gross margin trung bình theo segment:")
print(result)
print(f"\nSegment cao nhất: {result.idxmax()} ({result.max():.4f})")

[1] Dữ liệu gốc: 2412 dòng
product_id      0
product_name    0
category        0
segment         0
size            0
color           0
price           0
cogs            0
dtype: int64

[2] Dòng vi phạm cogs >= price: 0

[3] Sau dropna: 2412 dòng

[4] Gross margin trung bình theo segment:
segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: gross_margin, dtype: float64

Segment cao nhất: Standard (0.3134)


CÂU 3: Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join
returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving returns.csv to returns (2).csv


In [ ]:
import pandas as pd

df_raw2 = pd.read_csv('orders.csv')  # bản gốc, không đụng vào

# Làm việc với bản copy
df = df_raw2.copy()

In [ ]:
# Load
returns = pd.read_csv('returns.csv')
products = pd.read_csv('products.csv')
returns_raw = returns.copy()
products_raw = products.copy()

print(f"[1] returns: {len(returns)} dòng")
print(f"    products: {len(products)} dòng")
print("\nreturns null:\n", returns.isnull().sum())
print("\nproducts null:\n", products.isnull().sum())

[1] returns: 39939 dòng
    products: 2412 dòng

returns null:
 return_id          0
order_id           0
product_id         0
return_date        0
return_reason      0
return_quantity    0
refund_amount      0
dtype: int64

products null:
 product_id      0
product_name    0
category        0
segment         0
size            0
color           0
price           0
cogs            0
dtype: int64


In [ ]:
# Drop thiếu dữ liệu các cột cần thiết
returns = returns.dropna(subset=['product_id', 'return_reason'])
products = products.dropna(subset=['product_id', 'category'])
print(f"\n[2] Sau dropna - returns: {len(returns)}, products: {len(products)}")

# Lọc sản phẩm Streetwear từ products
streetwear = products[products['category'] == 'Streetwear'][['product_id']]
print(f"\n[3] Số sản phẩm Streetwear: {len(streetwear)}")

# Join returns với streetwear products
merged = returns.merge(streetwear, on='product_id', how='inner')
print(f"\n[4] Sau join (chỉ giữ Streetwear): {len(merged)} dòng")
print(merged['return_reason'].value_counts())

# Lý do phổ biến nhất
top_reason = merged['return_reason'].value_counts().idxmax()
print(f"\n[5] Lý do trả hàng nhiều nhất: {top_reason}")


[2] Sau dropna - returns: 39939, products: 2412

[3] Số sản phẩm Streetwear: 1320

[4] Sau join (chỉ giữ Streetwear): 21799 dòng
return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64

[5] Lý do trả hàng nhiều nhất: wrong_size


Câu 4:  Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung
bình (bounce_rate) thấp nhấttrên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving web_traffic.csv to web_traffic (2).csv


In [ ]:
import pandas as pd

df_raw3 = pd.read_csv('orders.csv')  # bản gốc, không đụng vào

# Làm việc với bản copy
df = df_raw3.copy()

In [ ]:
import pandas as pd

df = pd.read_csv('web_traffic.csv')
df_raw = df.copy()
print(f"[1] Dữ liệu gốc: {len(df)} dòng")
print(df.isnull().sum())
print(f"\nCác traffic_source: {df['traffic_source'].unique()}")

# Drop thiếu dữ liệu các cột cần thiết
df = df.dropna(subset=['traffic_source', 'bounce_rate'])
print(f"\n[2] Sau dropna: {len(df)} dòng")

# Kiểm tra bounce_rate hợp lệ (0 đến 1 hoặc 0 đến 100)
print(f"\n[3] bounce_rate range: {df['bounce_rate'].min()} - {df['bounce_rate'].max()}")

# Tính bounce_rate trung bình theo traffic_source
result = df.groupby('traffic_source')['bounce_rate'].mean().sort_values()
print(f"\n[4] Bounce rate trung bình theo nguồn:")
print(result)

print(f"\nNguồn có bounce_rate thấp nhất: {result.idxmin()} ({result.min():.4f})")

[1] Dữ liệu gốc: 3652 dòng
date                        0
sessions                    0
unique_visitors             0
page_views                  0
bounce_rate                 0
avg_session_duration_sec    0
traffic_source              0
dtype: int64

Các traffic_source: ['organic_search' 'direct' 'referral' 'social_media' 'paid_search'
 'email_campaign']

[2] Sau dropna: 3652 dòng

[3] bounce_rate range: 0.0032 - 0.0058

[4] Bounce rate trung bình theo nguồn:
traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64

Nguồn có bounce_rate thấp nhất: email_campaign (0.0045)


Câu 4: Tỷ ệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id
không null) xấp xỉ là bao nhiêu?

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving order_items.csv to order_items.csv


In [ ]:
import pandas as pd

df_raw4 = pd.read_csv('order_items.csv')  # bản gốc, không đụng vào

# Làm việc với bản copy
df = df_raw4.copy()

/tmp/ipykernel_49636/3764098512.py:3: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw4 = pd.read_csv('order_items.csv')  # bản gốc, không đụng vào


In [ ]:
import pandas as pd

df = pd.read_csv('order_items.csv', low_memory=False)
total_original = len(df)
print(f"Ban đầu: {total_original} dòng")

# Bước 1: Drop duplicate theo cặp (order_id, product_id), giữ dòng đầu
df = df.drop_duplicates(subset=['order_id', 'product_id'], keep='first')
print(f"Sau bước 1 (dedup order+product): {len(df)} dòng")

# Bước 2: Bỏ dòng mà CẢ HAI promo_id VÀ promo_id_2 đều null
df = df[~(df['promo_id'].isna() & df['promo_id_2'].isna())]
print(f"Sau bước 2 (bỏ dòng không có promo nào): {len(df)} dòng")

# Tỷ lệ = số dòng còn lại / số dòng BAN ĐẦU
pct = len(df) / total_original * 100
print(f"\nTỷ lệ %: {pct:.2f}%")

Ban đầu: 714669 dòng
Sau bước 1 (dedup order+product): 714653 dòng
Sau bước 2 (bỏ dòng không có promo nào): 276309 dòng

Tỷ lệ %: 38.66%
